# Pacotes

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from imblearn.under_sampling import RandomUnderSampler
from sklearn import  model_selection

# Funções

In [2]:
#################################
###### IV and WoE Discretas #####
#################################

def Woe_IV(df, feature, target):
    df_woe_iv = (pd.crosstab(round(df[feature],0),df[target],
                      normalize='columns')
             .assign(RR=lambda dfx: dfx[1] / dfx[0])
             .assign(woe=lambda dfx: np.log(dfx[1] / dfx[0]))
             .assign(iv=lambda dfx: (dfx['woe']*(dfx[1]-dfx[0]))).replace([np.inf, -np.inf], 0)
             .assign(iv_total=lambda dfx: np.sum(dfx['iv'])))
    

    return df_woe_iv

In [3]:
#################################
###### IV and WoE Contínuas #####
#################################


def Woe_IV_cont(df, features, target):
    
    aux = features + [target] 
    
    df = df[aux].copy()
    
    # Dataframe vazio
    df_woe_iv = pd.DataFrame({},index=[])
    
    # Quantidade de valores com ocorrencia
    _t1 = sum(df[target])
    # Quantidade de valores sem ocorrencia
    _t0 =  len(df[target]) - _t1
    
    # Percentil das variáveis contínuas
    quantil = df.iloc[:, :-1].quantile([.1, .2, .3, .4, .5, .6, .7, .8, .9], axis = 0)
    
    
    for coluna in quantil.columns:
   
        # Valores não duplicados do limite dos quantis
        lista_aux = quantil[[coluna]].drop_duplicates().to_numpy()
    
        _tiv = 0
        
        for q in range(len(lista_aux)):
            
            
            if q>0:
                localizacao = df[(df.loc[:,coluna] > float(lista_aux[q-1])) & (df.loc[:,coluna] <= float(lista_aux[q]))].index
                limite = str(lista_aux[q-1]) + ' a ' + str(lista_aux[q])
            else:
                localizacao = df[(df.loc[:,coluna] <= float(lista_aux[q]))].index
                limite = '<=' + str(lista_aux[q])
                
            populacao = len(localizacao)  
            
            # Com ocorrência
            _1 = sum(df.loc[localizacao,target])
            _p1 = _1/_t1
            
            # Sem ocorrência
            _0 = populacao - _1
            _p0 = _0/_t0
            
            # Risco relativo
            if _p1 == 0 or _p0 == 0:
                _RR = 1
            else:
                _RR = _p1/_p0
            
            # Weight of evidence
            _woe = np.log(_RR)
            
            # Information value
            _iv = round(_woe*(_p1-_p0),2)
            
            # Information value total
            _tiv = _tiv+_iv
                    
            
            dframe = pd.DataFrame({'variavel':coluna , 'limite':limite , '0': _p0 , '1': _p1, 'RR':_RR, 'WoE': _woe , 'IV':  _iv}
                                  , index = [coluna])  
            
            df_woe_iv = pd.concat([df_woe_iv, dframe], ignore_index=True)
            
        dframe = pd.DataFrame({'variavel':coluna ,'limite': ' ' , '0': 1 , '1': 1, 'RR': 1 , 'WoE': 0 , 'IV':  _tiv}
                                  , index =[coluna])
        
        df_woe_iv = pd.concat([df_woe_iv, dframe], ignore_index=True)
            
    return df_woe_iv


In [4]:
#################################
###### Top ABS Correlations #####
#################################


def get_redundant_pairs(df):
    '''Get diagonal and lower triangular pairs of correlation matrix'''
    pairs_to_drop = set()
    cols = df.columns
    for i in range(0, df.shape[1]):
        for j in range(0, i+1):
            pairs_to_drop.add((cols[i], cols[j]))
    return pairs_to_drop

def get_top_abs_correlations(df, n=5):
    au_corr = df.corr().abs().unstack()
    labels_to_drop = get_redundant_pairs(df)
    au_corr = au_corr.drop(labels=labels_to_drop).sort_values(ascending=False)
    return au_corr[0:n]

In [5]:
#####################
###### Fill NaN #####
#####################

def Fill_NaN(data_frame):
    
    newdf = data_frame.select_dtypes(include='number')
    
    for i in newdf:
        data_frame[i] = data_frame[i].transform(lambda x: x.fillna(0))


In [6]:
########################
###### Normalizador ####
########################
def normalizador(dados):

    dados2= deepcopy(dados)
    
    for j in range(dados2.shape[1]):
        menor = min(dados2.iloc[:,j]) 
        maior = max(dados2.iloc[:,j])
        
        dif = maior - menor
        if dif==0:
            dif=1 
        
        for i in range(dados2.shape[0]):
            dados2.iloc[i,j] = (dados2.iloc[i,j]-menor)/ dif

    return(dados2) 

# Importação dos dados

In [ ]:
dados = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df_final_4.csv'
                     ,sep = ','
                     ,decimal='.'
                      )

# Análise da qualidade das variáveis

In [ ]:
#Informações gerais dos dados

dados.info()

In [ ]:
#Descrição das variáveis numéricas

pd.set_option('display.max_columns', None)
dados.describe()

In [ ]:
# Percentual de ocorrência sem chuva
colunas = ['quinzemin', 
'trintamin',   
'umahora',   
'seishoras',      
'dozehoras',      
'vintequatrohoras',   
'quarentaoitohoras',     
'setentaduashoras',     
'noventaseishoras',     
'mes'
,'chuva_d2_quinzemin'	
,'chuva_d2_trintamin'	
,'chuva_d2_umahora'	
,'chuva_d2_seishoras'	
,'chuva_d2_dozehoras'	
,'chuva_d2_vintequatrohoras'	
,'chuva_d2_quarentaoitohoras'	
,'chuva_d2_setentaduashoras'	
,'chuva_d2_noventaseishoras'	
,'chuva_d2_mes'	
,'chuva_d3_quinzemin'	
,'chuva_d3_trintamin'	
,'chuva_d3_umahora'	
,'chuva_d3_seishoras'	
,'chuva_d3_dozehoras'	
,'chuva_d3_vintequatrohoras'	
,'chuva_d3_quarentaoitohoras'	
,'chuva_d3_setentaduashoras'	
,'chuva_d3_noventaseishoras'	
,'chuva_d3_mes'	
,'chuva_d4_quinzemin'	
,'chuva_d4_trintamin'	
,'chuva_d4_umahora'	
,'chuva_d4_seishoras'	
,'chuva_d4_dozehoras'	
,'chuva_d4_vintequatrohoras'	
,'chuva_d4_quarentaoitohoras'	
,'chuva_d4_setentaduashoras'	
,'chuva_d4_noventaseishoras'	
,'chuva_d4_mes'		
,'chuva_d5_quinzemin'	
,'chuva_d5_trintamin'	
,'chuva_d5_umahora'	
,'chuva_d5_seishoras'	
,'chuva_d5_dozehoras'	
,'chuva_d5_vintequatrohoras'	
,'chuva_d5_quarentaoitohoras'	
,'chuva_d5_setentaduashoras'	
,'chuva_d5_noventaseishoras'	
,'chuva_d5_mes'  ]
for i in colunas:
  print('Percentual de ocorrências com ' + i +' acumulado sem chuva: ' + repr(round(sum(dados[i]==0)/len(dados)*100,2)) + '%' )

del(colunas)
del(i)


In [ ]:
# Valores únicos por variável

pd.options.display.max_rows = None
print('Valores únicos em cada variável: \n')
dados.nunique(dropna=False).sort_values()
#pd.options.display.max_rows = 10

In [ ]:
# Valores ausentes

df_null = dados.isnull().mean(axis = 0)
df_null = df_null[df_null > 0] * 100
print("Colunas com valores ausentes (qtd relativa): \n\n{}\n".format(df_null.sort_values(axis=0, ascending=False)))

del(df_null)

In [ ]:
pd.options.display.max_rows = 10

In [ ]:
colunas_excluir = ['flg_rocha']
dados = dados.drop(colunas_excluir, axis=1)

# Seleção de variáveis via estudo IV e WoE

In [ ]:
list(dados.columns)

## Variáveis Categóricas

In [ ]:
variaveis = [
 'Declividade_classes_A',
 'Declividade_classes_B',
 'Declividade_classes_C',
 'Orientacao_octantes',
 'Curv_Vertical_3classes',
 'Curv_Vertical_5classes',
 'Curv_Horizontal_3classes',
 'Curv_Horizontal_5classes',
 'Forma_de_terreno_classes',
 'ADD_divisores_talvegues',
 'flg_comunidades',
 'flg_agricola',
 'flg_exploracao_mineral',
 'flg_corpo_hidrico',
 'flg_cobertura_vegetal',
 'flg_afloramento_rochoso',
 'flg_favela',
 'flg_ocupacao_desordenada',
 'flg_areas_de_risco',
 'Acoes_DC_0',
 'Acoes_DC_1',
 'Acoes_DC_2',
 'Acoes_DC_3',
 'graurisc',
 'mes_ocorrencia',
    'Vegetacao_Natural_Dominante',
 'Area_Antropica_Dominante',
 'Floresta_Ombrofila_Densa',
 'Formacao_Pioneira',
 'Floresta_Ombrofila_Densa_Submontana',
 'Formacao_Pioneira_com_influencia_fluviomarinha',
 'Influencia_urbana',
 'Vegetacao_Secundaria',
 'Argilossolo',
 'Gleissolo',
 'Argilossolo_Amarelo',
 'Argilossolo_Vermelho_Amarelo',
 'Gleissolo_Melanico',
 'Area_Urbana',
 'Unidades_de_Conservacao_Protecao_N_Integral',
 'Unidades_de_Conservacao_Protecao_Integral',
 'Sem_Plano_de_Menejo',
 'Plano_de_Manejo'
]

In [ ]:
for v in variaveis:
    print('----------------------------')
    print(v)
    print()
    print(Woe_IV(dados, v , 'flg_ocorrencia'))
    print()
    print('----------------------------')
    print()


Preditividade das variáveis:
    
   Não útil para predição (IV<0,02)
   
	Curv_Horizontal_3classes
	Curv_Horizontal_5classes
	flg_agricola
	flg_exploracao_mineral
	flg_afloramento_rochoso
	flg_ocupacao_desordenada
	Acoes_DC_1
	Acoes_DC_2
	Acoes_DC_3
	Vegetacao_Natural_Dominante
	Floresta_Ombrofila_Densa_Submontana
	Formacao_Pioneira_com_influencia_fluviomarinha
	Argilossolo_Amarelo
	Unidades_de_Conservacao_Protecao_N_Integral
	Sem_Plano_de_Menejo
    
   Poder preditivo fraco (0,02<=IV<0,1)
   
	Orientacao_octantes
	Curv_Vertical_3classes
	Curv_Vertical_5classes
	Forma_de_terreno_classes
	ADD_divisores_talvegues
	flg_areas_de_risco
	Acoes_DC_0
	graurisc
	mes_ocorrencia
	Formacao_Pioneira
  

   Poder preditivo médio (0,1<=IV<0,3)
   
	Declividade_classes_B
	flg_favela
	Area_Antropica_Dominante
	Floresta_Ombrofila_Densa
	Influencia_urbana
	Vegetacao_Secundaria
	Gleissolo
	Gleissolo_Melanico
    
   Poder preditivo forte (0,3<=IV<0,5)
   
	Declividade_classes_A
	Declividade_classes_C
	flg_cobertura_vegetal
        
   Poder preditivo suspeito (IV>0,5)
    
	flg_comunidades
	Argilossolo
	Argilossolo_Vermelho_Amarelo
	Area_Urbana
	Unidades_de_Conservacao_Protecao_Integral
	Plano_de_Manejo

## Variáveis Numéricas

In [ ]:
variaveis2 = [
 'quinzemin',
 'trintamin',
 'umahora',
 'seishoras',
 'dozehoras',
 'vintequatrohoras',
 'quarentaoitohoras',
 'setentaduashoras',
 'noventaseishoras',
 'mes',
 'Altitude_numerica',
 'Declividade_numerica',
 'Orientacao_numerica',
 'Curv_Vertical_numerica',
 'Curv_Horizontal_numerica',
 'Relevo_sombreado_numerico',
'chuva_d2_quinzemin',
 'chuva_d2_trintamin',
 'chuva_d2_umahora',
 'chuva_d2_seishoras',
 'chuva_d2_dozehoras',
 'chuva_d2_vintequatrohoras',
 'chuva_d2_quarentaoitohoras',
 'chuva_d2_setentaduashoras',
 'chuva_d2_noventaseishoras',
 'chuva_d2_mes',
 'chuva_d3_quinzemin',
 'chuva_d3_trintamin',
 'chuva_d3_umahora',
 'chuva_d3_seishoras',
 'chuva_d3_dozehoras',
 'chuva_d3_vintequatrohoras',
 'chuva_d3_quarentaoitohoras',
 'chuva_d3_setentaduashoras',
 'chuva_d3_noventaseishoras',
 'chuva_d3_mes',
 'chuva_d4_quinzemin',
 'chuva_d4_trintamin',
 'chuva_d4_umahora',
 'chuva_d4_seishoras',
 'chuva_d4_dozehoras',
 'chuva_d4_vintequatrohoras',
 'chuva_d4_quarentaoitohoras',
 'chuva_d4_setentaduashoras',
 'chuva_d4_noventaseishoras',
 'chuva_d4_mes',
 'chuva_d5_quinzemin',
 'chuva_d5_trintamin',
 'chuva_d5_umahora',
 'chuva_d5_seishoras',
 'chuva_d5_dozehoras',
 'chuva_d5_vintequatrohoras',
 'chuva_d5_quarentaoitohoras',
 'chuva_d5_setentaduashoras',
 'chuva_d5_noventaseishoras',
 'chuva_d5_mes']

In [ ]:
pd.options.display.max_rows = None
Woe_IV_cont(dados, variaveis2, 'flg_ocorrencia'  )

Preditividade das variáveis:
    
   Não útil para predição (IV<0,02)
	
	quinzemin
	trintamin
	umahora
	seishoras
	dozehoras
	vintequatrohoras
	Orientacao_numerica
	Curv_Horizontal_numerica
	chuva_d2_quinzemin
	chuva_d2_trintamin
	chuva_d2_umahora
	chuva_d3_trintamin
	chuva_d3_umahora
	chuva_d3_quinzemin
	chuva_d4_quinzemin
	chuva_d4_trintamin
	chuva_d4_seishoras
	chuva_d4_umahora
	chuva_d5_quinzemin
	chuva_d5_trintamin
	chuva_d5_umahora
	chuva_d5_seishoras
	chuva_d5_dozehoras
	chuva_d5_quarentaoitohoras

    
   Poder preditivo fraco (0,02<=IV<0,1)
 
	vintequatrohoras  
	quarentaoitohoras
	setentaduashoras
	noventaseishoras
	mes
	Relevo_sombreado_numerico
	chuva_d2_seishoras
	chuva_d2_dozehoras
	chuva_d2_vintequatrohoras
	chuva_d2_quarentaoitohoras
	chuva_d2_mes
	chuva_d3_seishoras
	chuva_d3_dozehoras
	chuva_d3_vintequatrohoras
	chuva_d3_quarentaoitohoras
	chuva_d3_setentaduashoras
	chuva_d3_noventaseishoras
	chuva_d3_mes
	chuva_d4_dozehoras
	chuva_d4_vintequatrohoras
	chuva_d4_quarentaoitohoras
	chuva_d4_setentaduashoras
	chuva_d5_vintequatrohoras
	chuva_d5_setentaduashoras
	chuva_d5_noventaseishoras
	

   Poder preditivo médio (0,1<=IV<0,3)
	
	Declividade_numerica
	Curv_Vertical_numerica
	chuva_d2_setentaduashoras
	chuva_d2_noventaseishoras
	chuva_d4_noventaseishoras
	chuva_d4_mes
	chuva_d5_mes

   Poder preditivo forte (0,3<=IV<0,5)
   
	Altitude_numerica
        
   Poder preditivo suspeito (IV>0,5)
    


# Remoção das variáveis 

In [ ]:
apagar = [
    'vintequatrohoras'  
,'quarentaoitohoras'
,'setentaduashoras'
,'noventaseishoras'
,'mes'
    ,'quinzemin'
,'trintamin'
,'umahora'
,'seishoras'
,'dozehoras'
,'vintequatrohoras'
,'Orientacao_numerica'
,'Curv_Horizontal_numerica'
,'chuva_d2_quinzemin'
,'chuva_d2_trintamin'
,'chuva_d2_umahora'
,'chuva_d3_trintamin'
,'chuva_d3_umahora'
,'chuva_d3_quinzemin'
,'chuva_d4_quinzemin'
,'chuva_d4_trintamin'
,'chuva_d4_seishoras'
,'chuva_d4_umahora'
,'chuva_d5_quinzemin'
,'chuva_d5_trintamin'
,'chuva_d5_umahora'
,'chuva_d5_seishoras'
,'chuva_d5_dozehoras'
,'chuva_d5_quarentaoitohoras'
,'Curv_Horizontal_3classes'
,'Curv_Horizontal_5classes'
,'flg_agricola'
,'flg_exploracao_mineral'
,'flg_afloramento_rochoso'
,'flg_ocupacao_desordenada'
,'Acoes_DC_1'
,'Acoes_DC_2'
,'Acoes_DC_3'
,'Vegetacao_Natural_Dominante'
,'Floresta_Ombrofila_Densa_Submontana'
,'Formacao_Pioneira_com_influencia_fluviomarinha'
,'Argilossolo_Amarelo'
,'Unidades_de_Conservacao_Protecao_N_Integral'
,'Sem_Plano_de_Menejo'
]


dados = dados.drop(apagar,axis=1)

# Correlação entre atributos

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(dados.corr(),annot=True)

In [ ]:
pd.options.display.max_rows = 100
print(get_top_abs_correlations(dados, 100))

Muito correlacionadas, retirar.
									
Gleissolo 0.258085	Gleissolo_Melanico 0.25808									
Argilossolo 0.616962	Argilossolo_Vermelho_Amarelo 0.602503	Area_Urbana 1.01877								
mes 0.06	chuva_d2_mes 0.05	chuva_d3_mes 0.06	chuva_d4_mes 0.14	chuva_d5_mes 0.13						
quarentaoitohoras 0.05	setentaduashoras 0.04	noventaseishoras 0.05								
Unidades_de_Conservacao_Protecao_Integral 0.866182	Plano_de_Manejo 0.843551									
chuva_d2_dozehoras 0.03	chuva_d2_noventaseishoras 0.14	chuva_d2_quarentaoitohoras 0.07	chuva_d2_setentaduashoras 0.12	chuva_d2_vintequatrohoras 0.07	chuva_d3_setentaduashoras 0.06	chuva_d3_quarentaoitohoras 0.06				
chuva_d4_noventaseishoras 0.1	chuva_d5_setentaduashoras 0.06	chuva_d4_setentaduashoras 0.04	chuva_d5_vintequatrohoras 0.05	 chuva_d4_quarentaoitohoras 0.03	chuva_d5_noventaseishoras 0.09	chuva_d4_dozehoras 0.03				
Declividade_classes_A 0.332987	Declividade_classes_B 0.266815	Declividade_numerica	Declividade_classes_C 0.397752							
flg_areas_de_risco 0.0502	graurisc 0.051458									
Influencia_urbana 0.20589	Vegetacao_Secundaria 0.118232									
chuva_d3_dozehoras 0.04	chuva_d3_noventaseishoras 0.05	chuva_d3_quarentaoitohoras 0.06	chuva_d3_seishoras 0.04	chuva_d3_setentaduashoras 0.06	chuva_d3_vintequatrohoras 0.06	chuva_d4_quarentaoitohoras 0.03	chuva_d4_noventaseishoras 0.10	chuva_d4_setentaduashoras 0.04		
Curv_Vertical_3classes 0.074197	Curv_Vertical_numerica 0.11	Curv_Vertical_5classes 0.076237								
Floresta_Ombrofila_Densa 0.133093	Formacao_Pioneira 0.056008									
flg_comunidades 0.647091	flg_favela 0.338029									
chuva_d2_seishoras 0.03	chuva_d2_dozehoras 0.03									
										



In [ ]:
apagar = ['Gleissolo_Melanico'
          ,'Argilossolo'
          ,'Argilossolo_Vermelho_Amarelo'
          ,'mes'
         ,'chuva_d2_mes'
         ,'chuva_d3_mes'
         ,'chuva_d5_mes'
          ,'Plano_de_Manejo'
          ,'setentaduashoras'
          ,'quarentaoitohoras'
#        ,'chuva_d2_dozehoras'
#        ,'chuva_d2_quarentaoitohoras'
#        ,'chuva_d3_setentaduashoras'
#        ,'chuva_d3_quarentaoitohoras'
#        ,'chuva_d4_quarentaoitohoras'
#        ,'chuva_d4_dozehoras'
         ,'Declividade_classes_A'
         ,'Declividade_classes_B'
         ,'flg_areas_de_risco'
         ,'Vegetacao_Secundaria'
#        ,'chuva_d3_dozehoras'
#        ,'chuva_d3_seishoras'
         ,'Curv_Vertical_numerica'
         ,'Formacao_Pioneira'
         ,'flg_favela'
#        ,'chuva_d2_seishoras'
         ]

dados = dados.drop(apagar,axis=1)

# Distribuição dos dados

In [ ]:
dados['flg_ocorrencia'].value_counts().plot(kind = 'pie', explode = [0, 0.1], figsize = (6, 6), autopct = '%1.1f%%', shadow = True)
plt.ylabel('Distribuição entre as classes')
plt.legend(['Negativa', 'Positiva'])
plt.show()

Uma pequena parte dos dados da amostra são com ocorrência. Fazer modelo de previsão assumindo que será modelo para eventos raros.

In [ ]:
dados.to_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df_final_trat.csv', index=False)

In [7]:
dados = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df_final_trat.csv'
                     ,sep = ','
                     ,decimal='.'
                      )

In [8]:
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 718446 entries, 0 to 718445
Data columns (total 46 columns):
 #   Column                                     Non-Null Count   Dtype  
---  ------                                     --------------   -----  
 0   flg_ocorrencia                             718446 non-null  float64
 1   id_tempo_x                                 718446 non-null  int64  
 2   id_tempo_y                                 331670 non-null  float64
 3   id_solicitacao                             331670 non-null  float64
 4   chuva_d2_seishoras                         718446 non-null  float64
 5   chuva_d2_dozehoras                         718446 non-null  float64
 6   chuva_d2_vintequatrohoras                  718446 non-null  float64
 7   chuva_d2_quarentaoitohoras                 718446 non-null  float64
 8   chuva_d2_setentaduashoras                  718446 non-null  float64
 9   chuva_d2_noventaseishoras                  718446 non-null  float64
 10  chuva_d3

In [10]:
sum(dados['flg_ocorrencia'])

724.0

# Separando as amostras

## Out Of Time - OOT

São as amostras de teste simulando o tempo futuro, isto é o que ainda não foi visto.


In [11]:
print(max(dados['id_tempo_x'])) #2021-12-20 23:30:00.000
print(min(dados['id_tempo_x'])) #2019-01-05 23:30:00.000

13225726
11491171


In [12]:
# Janela deslizante
# Retirar os últimos 2 meses para teste OOT

_2m = []

for i in range(2, 40,2):
    
    v = min(dados['id_tempo_x']) + (i)*30*24*60 
    
    if v >  max(dados['id_tempo_x']) :
        break 
        
    _2m.append(v)
    

In [13]:
_2m

[11577571,
 11663971,
 11750371,
 11836771,
 11923171,
 12009571,
 12095971,
 12182371,
 12268771,
 12355171,
 12441571,
 12527971,
 12614371,
 12700771,
 12787171,
 12873571,
 12959971,
 13046371,
 13132771]

In [14]:
for i in range(len(_2m)-1):
    df_2m = dados[(dados.loc[:,'id_tempo_y' ] <= _2m[i+1]) & (dados.loc[:,'id_tempo_y' ] > _2m[i])]
    df_2m.to_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+str(i)+'_final_oot.csv', index=False)

## Out of Sample - OOS

Dados dentro do tempo da amostra de treinamento para o teste.



In [16]:
X_treino, X_teste, y_treino, y_teste, idx_treino, idx_teste = model_selection.train_test_split(dados.drop(['flg_ocorrencia'],axis=1)
                                                                                               , dados['flg_ocorrencia']
                                                                                               , dados.index
                                                                                               , test_size =1/3
                                                                                               ,random_state= 42
                                                                                               #, stratify = dados['flg_comunidades']
                                                                                              )
print(X_treino.shape)
print(X_teste.shape)
print(y_treino.shape)
print(y_teste.shape)

(478964, 45)
(239482, 45)
(478964,)
(239482,)


In [17]:
X_teste['flg_ocorrencia']=y_teste

In [18]:
for i in range(len(_2m)-1):
    
    df_teste = X_teste.drop(X_teste[X_teste.loc[:,'id_tempo_x' ] > _2m[i]].index) 
    df_teste.to_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+str(i)+'_oos.csv', index=False)

## In Sample - INS

Dados que serão utilizados para o treinamento do modelo.




In [19]:
X_treino['flg_ocorrencia']=y_treino

In [20]:
n_colunas=len(X_treino.columns)

### Over Sampling

Utilizado o Balanceamento SMOTE(Synthetic Minority Oversampling Technique)  - 2002, Nitesh Chawla, et al.

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
oversample = SMOTE(sampling_strategy='minority'
                   ,k_neighbors = 3
                   ,random_state=42)

In [ ]:
Fill_NaN(X_treino)

In [ ]:
for i in range(len(_2m)-1):
    
    df_over = X_treino.drop(X_treino[X_treino.loc[:,'id_tempo_x' ] > _2m[i]].index)
    
    if round(sum(df_over.iloc[:,(n_colunas-1)])) <= 3 :
        continue

    X_over, y_over = oversample.fit_resample(df_over.iloc[:,0:(n_colunas-1)], df_over.iloc[:,(n_colunas-1)])
    
    X_over['flg_ocorrencia']=y_over
    
    X_over.to_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+str(i)+'_ins_over.csv', index=False)

In [ ]:
df_sc = X_treino[X_treino.loc[:,'flg_comunidades' ] == 0]
df_cc = X_treino[X_treino.loc[:,'flg_comunidades' ] == 1]

for i in range(len(_2m)-1):
    
    df_oversc = df_sc.drop(df_sc[df_sc.loc[:,'id_tempo_x' ] > _2m[i]].index)
    
    if round(sum(df_oversc.iloc[:,(n_colunas-1)])) <= 3 :
        continue
    
    X_oversc, y_oversc = oversample.fit_resample(df_oversc.iloc[:,0:(n_colunas-1)], df_oversc.iloc[:,(n_colunas-1)])
    
    X_oversc['flg_ocorrencia']=y_oversc
    
    
    df_overcc = df_cc.drop(df_cc[df_cc.loc[:,'id_tempo_x' ] > _2m[i]].index)

    if round(sum(df_overcc.iloc[:,(n_colunas-1)])) <= 3 :
        continue

    
    X_overcc, y_overcc = oversample.fit_resample(df_overcc.iloc[:,0:(n_colunas-1)], df_overcc.iloc[:,(n_colunas-1)])
    
    X_overcc['flg_ocorrencia']=y_overcc
    
    X_over = pd.concat([X_oversc, X_overcc], ignore_index=True)
    
    
    X_over.to_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+str(i)+'_ins_c_over.csv', index=False)

### Under e Over sampling



In [ ]:
oversample = SMOTE(sampling_strategy= 0.25
                    , k_neighbors = 3
                    , random_state=42)

undersample = RandomUnderSampler(sampling_strategy='majority'
                                 , random_state=42
                                 , replacement=False)

In [ ]:
df_uo.iloc[:,(n_colunas-1)]

In [ ]:
for i in range(len(_2m)-1):
    
    df_uo = X_treino.drop(X_treino[X_treino.loc[:,'id_tempo_x' ] > _2m[i]].index)
   
    if round(sum(df_uo.iloc[:,(n_colunas-1)])) <= 3 :
        continue
    
    X_o, y_o = oversample.fit_resample(df_uo.iloc[:,0:(n_colunas-1)], df_uo.iloc[:,(n_colunas-1)])
    
    X_combined_sampling, y_combined_sampling = undersample.fit_resample(X_o, y_o)
    
    X_combined_sampling['flg_ocorrencia'] = y_combined_sampling
    
    X_combined_sampling.to_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+str(i)+'_ins_uo.csv', index=False)

## Sem reamostragem

In [21]:
for i in range(len(_2m)-1):
    
    df_over = X_treino.drop(X_treino[X_treino.loc[:,'id_tempo_x' ] > _2m[i]].index)
    df_over.to_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+str(i)+'_ins.csv'
                      , index=False
                      , header=True
                      , decimal='.'
                      , sep=',')